In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch
device= "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Visualizations and Text Cleaning

In [3]:
import re
import matplotlib.pyplot as plt

train= pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


In [ ]:
answer_counts = train["answer"].value_counts()
plt.figure(figsize=(7, 4))
plt.bar(answer_counts.index, answer_counts.values)
plt.xlabel("Correct Answer Option")
plt.ylabel("Number of Questions")
plt.title("Distribution of Correct Answer Options")
plt.show()

q_lengths = train["prompt"].head(6).astype(str).str.split().str.len()
plt.figure(figsize=(8, 4))
plt.bar([f"Q {i}" for i in range(1, 7)], q_lengths.values)
plt.xlabel("Question")
plt.ylabel("Number of Words")
plt.title("Question Length for First 6 Questions")
plt.show()

In [4]:
print(train.isnull().sum().sum())  #no nulls values

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\-\./\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["cleaned_prompt"] = train["prompt"].apply(clean_text)
for opt in ["A","B","C","D","E"]:
    train[f"cleaned_{opt}"] = train[f"{opt}"].apply(clean_text)
train.head(5)

0


,id,prompt,A,B,C,D,E,answer,cleaned_prompt,cleaned_A,cleaned_B,cleaned_C,cleaned_D,cleaned_E
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


# Neural network from scratch


In [ ]:
# !pip install wandb

In [ ]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("wandb")
# wandb.login()

In [7]:
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb

torch.manual_seed(42)
np.random.seed(42)
train_df, test_df = train_test_split(train, test_size=0.2,random_state=42, stratify=train["answer"])

options = ["A", "B", "C", "D", "E"]
def create_pairs(df):
    texts = []
    labels = []
    for _, row in df.iterrows():
        prompt = row["cleaned_prompt"]
        answer = row["answer"]
        for opt in options:
            text = prompt + " " + row[f"cleaned_{opt}"]
            texts.append(text)
            if answer == opt:
                labels.append(1)
            else:
                labels.append(0)
    return texts, labels
train_texts, train_labels = create_pairs(train_df)
test_texts, test_labels = create_pairs(test_df)

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())
    
word_counts = Counter()
for text in train_texts:
    word_counts.update(tokenize(text))
word_to_id = {"<PAD>": 0, "<UNK>": 1}
for word in word_counts:
    word_to_id[word] = len(word_to_id)

def text_to_ids(text):
    tokens = tokenize(text)
    return [word_to_id.get(word, word_to_id["<UNK>"]) for word in tokens]
train_sequence = [text_to_ids(text) for text in train_texts]
test_sequence = [text_to_ids(text) for text in test_texts]

max_len = min(128, max(len(seq) for seq in train_sequence))
def pad_seq(seq, max_len):
    if len(seq) > max_len:
        return seq[:max_len]
    return seq + [0] * (max_len - len(seq))
X_train = np.array([pad_seq(seq, max_len) for seq in train_sequence])
X_test = np.array([pad_seq(seq, max_len) for seq in test_sequence])
y_train = np.array(train_labels)
y_test = np.array(test_labels)

class MCQDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = MCQDataset(X_train, y_train)
test_dataset = MCQDataset(X_test,y_test)

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

class NN(nn.Module):
    def __init__(self, vocab_size):
        super(NN, self).__init__()
        self.embedding = nn.Embedding(vocab_size,128,padding_idx=0)
        self.fc1 = nn.Linear(128,64)
        self.fc2 = nn.Linear(64,2)
    def forward(self, x):
        embds = self.embedding(x)
        mask = (x != 0).unsqueeze(-1)
        embds = embds * mask
        sum = embds.sum(dim=1)
        count = mask.sum(dim=1).clamp(min=1)
        x = sum / count
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# wandb.init(project="dlgenai-t226",
#             name="Neural Network",
#             config={"Model": "NN", "Epochs": 5, "Random Seed": 42,})
model = NN(vocab_size=len(word_to_id)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs,y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

#     wandb.log({"epoch": epoch + 1,
#             "train_loss": avg_loss })
# wandb.finish()
    
model.eval()
all_outputs = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        all_outputs.append(outputs.cpu())

test_outputs = torch.cat(all_outputs)
y_pred = torch.argmax(test_outputs,dim=1).numpy()
probs = torch.softmax(test_outputs,dim=1).numpy()
positive_probs = probs[:, 1]
pred_opt = []
true_ans = []
for i in range(0,len(positive_probs),5):
    q_probs = positive_probs[i:i+5]
    ranked_indices = np.argsort(q_probs)[::-1]
    top3 = [options[j] for j in ranked_indices[:3]]
    pred_opt.append(top3)
    correct_index = np.argmax(y_test[i:i+5])
    true_ans.append(options[correct_index])

def map3(true_ans, pred_options):
    score = 0
    for actual, preds in zip(true_ans,pred_options):
        if actual in preds:
            score += 1 / (preds.index(actual) + 1)
    return score / len(true_ans)
map3_score = map3(true_ans,pred_opt)
print("MAP@3:", map3_score)

opt1 = [p[0]for p in pred_opt]
acc_n = accuracy_score(true_ans,opt1)
f1_n = f1_score(true_ans,opt1,average="macro")
print("Accuracy:", acc_n)
print("F1 Score:", f1_n)

MAP@3: 0.9291666666666667
Accuracy: 0.8775
F1 Score: 0.8756438637932842


In [ ]:
# wandb.init(
#     project="dlgenai-t226",
#     name="Neural Network",
#     config={"Model": "NN",
#             "Epochs": 5,
#             "Random Seed": 42,})

# wandb.log({"accuracy": acc_n,
#             "f1_score": f1_n,
#             "map@3": map3_score })
# wandb.finish()

# RoBERTa tokenization & training

In [5]:
import random
random.seed(42)
torch.cuda.manual_seed_all(42)

In [8]:
from transformers import AutoTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

def create_pairs(df):
    rows = []
    for _, row in df.iterrows():
        prompt = row["cleaned_prompt"]
        for opt in options:
            rows.append({"question_id": row["id"],
                    "text": prompt + " </s> " + row[f"cleaned_{opt}"],
                    "label": 1 if row["answer"] == opt else 0, "option": opt})
    return pd.DataFrame(rows)
train_pairs = create_pairs(train_df)
test_pairs = create_pairs(test_df)
train_dataset = Dataset.from_pandas(train_pairs)
val_dataset = Dataset.from_pandas(test_pairs)

In [9]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True,padding="max_length",max_length=256 )
    
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])
val_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

# wandb.init(project="dlgenai-t226",
#             name="roberta-pretrained",
#            config={"Model": "RoBERTa",
#                     "Epochs": 3,
#                     "Batch Size": 16,
#                     "Learning Rate": 2e-5} )
           
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
training_args = TrainingArguments(output_dir="./roberta_results",
                            eval_strategy="epoch",
                            save_strategy="epoch",
                            learning_rate=2e-5,
                            per_device_train_batch_size=16,
                            per_device_eval_batch_size=16,
                            num_train_epochs=4,
                            weight_decay=0.01,
                            logging_steps=100,
                            load_best_model_at_end=True, seed=42,
                            # report_to="wandb"
                            )

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()

trainer.save_model("./roberta_model")
tokenizer.save_pretrained("./roberta_model")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./roberta_model/tokenizer_config.json', './roberta_model/tokenizer.json')

In [13]:
!cd /kaggle/working && zip -r roberta_model.zip roberta_model

  adding: roberta_model/ (stored 0%)
  adding: roberta_model/config.json (deflated 51%)
  adding: roberta_model/training_args.bin (deflated 53%)
  adding: roberta_model/tokenizer_config.json (deflated 50%)
  adding: roberta_model/model.safetensors (deflated 15%)
  adding: roberta_model/tokenizer.json (deflated 82%)


In [ ]:
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy:", acc)
print("F1 Score:", f1)

probs = torch.softmax(torch.tensor(preds.predictions), dim=1).numpy()
positive_probs = probs[:, 1]
rows_per_q = 5
pred_opt = []
true_ans = []   
for i in range(0, len(positive_probs), rows_per_q):
    q_probs = positive_probs[i:i+5]
    ranked_indices = np.argsort(q_probs)[::-1]
    top3 = [options[j] for j in ranked_indices[:3]]
    pred_opt.append(top3)
    correct_index = np.argmax(y_true[i:i+5])
    true_ans.append(options[correct_index])

map3_r = map3(true_ans, pred_opt)
print("MAP@3:", map3_r)

In [ ]:
# wandb.init(project="dlgenai-t226",
#             name="roberta-pretrained",
#            config={"Model": "RoBERTa",
#                     "Epochs": 3,
#                     "Batch Size": 16,
#                     "Learning Rate": 2e-5} )
# wandb.log({ "accuracy": acc,
#             "f1_score": f1,
#              "map@3": map3_r})
# wandb.finish()

# Sentence transformer

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")
pred_opt = []
true_ans = []

for _, row in test_df.iterrows():
    prompt = row["cleaned_prompt"]
    opt_texts = [row["cleaned_A"], row["cleaned_B"],row["cleaned_C"],row["cleaned_D"], row["cleaned_E"] ]
    prompt_embd = model.encode([prompt])
    opt_embd = model.encode(opt_texts)
    scores = cosine_similarity(prompt_embd, opt_embd)[0]
    ranked = np.argsort(scores)[::-1]
    top3 = [options[i] for i in ranked[:3]]
    pred_opt.append(top3)
    true_ans.append(row["answer"])

map3_s = map3(true_ans, pred_opt)
print("MAP@3:", map3_s)

opt1 = [p[0] for p in pred_opt]
acc_s = accuracy_score(true_ans, opt1)
print("Accuracy:", acc_s)

f1_s = f1_score(true_ans,opt1, average="macro")
print("F1 Score:", f1_s)

In [ ]:
# wandb.init( project="dlgenai-t226",
#             name="SentenceTransformer",
#             config={"Model": "Sentence Transformer",
#                     "Embedding Model": "all-MiniLM-L6-v2",
#                     "Similarity Metric": "Cosine Similarity"})
# wandb.log({"accuracy": acc_s,
#             "f1_score": f1_s,
#             "map@3": map3_s})
# wandb.finish()

# Models comparison

In [ ]:
print("Neural Network:")
print("    Accuracy:", acc_n)
print("    F1:", f1_n)
print("    MAP@3:", map3_score)

print("Sentence Transformer:")
print("    Accuracy:", acc_s)
print("    F1:", f1_s)
print("    MAP@3:", map3_s)

print("RoBERTa:")
print("    Accuracy:", acc)
print("    F1:", f1)
print("    MAP@3:", map3_r)

# Prediction on test data with highest MAP@3 model

In [ ]:
# RoBERTa has the highest MAP@3 score
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test["cleaned_prompt"] = test["prompt"].apply(clean_text)
for opt in ["A", "B", "C", "D", "E"]:
    test[f"cleaned_{opt}"] = test[opt].apply(clean_text)
    
rows = []
for _, row in test.iterrows():
    prompt = row["cleaned_prompt"]
    for opt in options:
        rows.append({"question_id": row["id"],
                    "text": prompt + " </s> " + row[f"cleaned_{opt}"],
                    "option": opt})
        
test_pairs = pd.DataFrame(rows)
test_dataset = Dataset.from_pandas(test_pairs)
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

preds = trainer.predict(test_dataset)
probs = torch.softmax(torch.tensor(preds.predictions),dim=1).numpy()
positive_probs = probs[:, 1]
predictions = []
for i in range(0, len(positive_probs), rows_per_q):
    q_probs = positive_probs[i:i+5]
    ranked = np.argsort(q_probs)[::-1]
    top3 = [options[j] for j in ranked[:3]]
    predictions.append(" ".join(top3))
    
submission = pd.DataFrame({"id": test["id"], "prediction": predictions})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission.head()

In [ ]:
# sample= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
# sample.to_csv('submission.csv', index=False)